# Neural Network Backpropagation Visualizer

An interactive notebook for exploring a small neural network **without autograd**.

### Architecture

$$
(x_1,x_2) \rightarrow z^{(1)} \rightarrow a^{(1)}
\rightarrow z^{(2)} \rightarrow a^{(2)}
\rightarrow z^{(3)} \rightarrow \operatorname{softmax}(z^{(3)})
$$

Each hidden layer contains exactly two neurons, so its representation can be plotted in two dimensions.

### Learning goals

- inspect every forward-pass value;
- follow gradients through every layer;
- manually move weights and biases;
- apply one gradient-descent update;
- observe how the representation and decision boundary change.

## 1. Setup

Run the next cell once. If a package is missing, uncomment the installation command.

In [1]:
# Colab setup
# Plotly is normally preinstalled. ipywidgets is used for the dashboard.
# Uncomment the next line only if an import fails:
# %pip install -q plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output, HTML

# Enable ipywidgets in Google Colab. This is harmless outside Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Use Colab's Plotly renderer for ordinary figure cells.
pio.renderers.default = "colab" if IN_COLAB else "notebook_connected"

np.set_printoptions(precision=4, suppress=True)
print("Environment:", "Google Colab" if IN_COLAB else "Jupyter")


Environment: Google Colab


## 2. XOR-like dataset

In [2]:
def create_xor_cloud(n_samples=500, margin=0.3, seed=42):
    """Create a reproducible 2D XOR-like point cloud."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, size=(n_samples, 2))

    for column in (0, 1):
        values = X[:, column]
        near_axis = np.abs(values) < margin
        signs = np.where(values[near_axis] >= 0, 1.0, -1.0)
        X[near_axis, column] = signs * (np.abs(values[near_axis]) + margin)

    y = np.logical_xor(X[:, 0] > 0, X[:, 1] > 0).astype(int)
    return X, y


X_data, y_data = create_xor_cloud()
print('X shape:', X_data.shape)
print('Class counts:', np.bincount(y_data))

X shape: (500, 2)
Class counts: [255 245]


In [3]:
dataset_fig = go.Figure()
for label, name in [(0, 'Class 0'), (1, 'Class 1')]:
    mask = y_data == label
    dataset_fig.add_trace(go.Scatter(
        x=X_data[mask, 0],
        y=X_data[mask, 1],
        mode='markers',
        name=name,
        marker={'size': 6, 'opacity': 0.65},
    ))

dataset_fig.update_layout(
    title='Original XOR-like dataset',
    xaxis_title='x₁',
    yaxis_title='x₂',
    height=500,
    template='plotly_white',
)

# Explicit rendering is required in Google Colab.
dataset_fig.show(renderer='colab' if IN_COLAB else None)


## 3. Activation functions

In [4]:
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))


def get_activation_fn(name):
    """Return activation and derivative with respect to the pre-activation z."""
    functions = {
        'Tanh': (np.tanh, lambda z: 1 - np.tanh(z) ** 2),
        'ReLU': (lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float)),
        'Sigmoid': (sigmoid, lambda z: sigmoid(z) * (1 - sigmoid(z))),
        'Leaky ReLU': (
            lambda z: np.where(z > 0, z, 0.01 * z),
            lambda z: np.where(z > 0, 1.0, 0.01),
        ),
    }
    if name not in functions:
        raise ValueError(f'Unknown activation: {name}')
    return functions[name]


def softmax(logits):
    logits = np.asarray(logits)
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)

## 4. Forward pass and cross-entropy loss

In [5]:
def forward_pass(x, W1, b1, W2, b2, W3, b3, activation_name='Tanh'):
    activation, _ = get_activation_fn(activation_name)

    z1 = x @ W1 + b1
    a1 = activation(z1)
    z2 = a1 @ W2 + b2
    a2 = activation(z2)
    z3 = a2 @ W3 + b3
    probabilities = softmax(z3)

    return {
        'x': x, 'z1': z1, 'a1': a1,
        'z2': z2, 'a2': a2,
        'z3': z3, 'probs': probabilities,
    }


def cross_entropy(probabilities, y_true):
    probabilities = np.asarray(probabilities)
    if probabilities.ndim == 1:
        return float(-np.log(probabilities[int(y_true)] + 1e-12))
    indices = np.arange(len(y_true))
    return float(np.mean(-np.log(probabilities[indices, y_true] + 1e-12)))

## 5. Manual backpropagation

In [6]:
def backward_pass(cache, y_true, W1, W2, W3, activation_name='Tanh'):
    """Analytical gradients for one selected sample."""
    _, activation_derivative = get_activation_fn(activation_name)
    x, z1, a1 = cache['x'], cache['z1'], cache['a1']
    z2, a2, probs = cache['z2'], cache['a2'], cache['probs']

    target = np.zeros(2)
    target[int(y_true)] = 1.0

    dL_dz3 = probs - target
    dL_dW3 = np.outer(a2, dL_dz3)
    dL_db3 = dL_dz3

    dL_da2 = dL_dz3 @ W3.T
    dL_dz2 = dL_da2 * activation_derivative(z2)
    dL_dW2 = np.outer(a1, dL_dz2)
    dL_db2 = dL_dz2

    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * activation_derivative(z1)
    dL_dW1 = np.outer(x, dL_dz1)
    dL_db1 = dL_dz1

    return {
        'dL_dz3': dL_dz3, 'dL_dW3': dL_dW3, 'dL_db3': dL_db3,
        'dL_da2': dL_da2, 'dL_dz2': dL_dz2,
        'dL_dW2': dL_dW2, 'dL_db2': dL_db2,
        'dL_da1': dL_da1, 'dL_dz1': dL_dz1,
        'dL_dW1': dL_dW1, 'dL_db1': dL_db1,
    }

## 6. Gradient check

In [7]:
def numerical_gradient(parameter, index, loss_fn, epsilon=1e-5):
    old_value = parameter[index]
    parameter[index] = old_value + epsilon
    loss_plus = loss_fn()
    parameter[index] = old_value - epsilon
    loss_minus = loss_fn()
    parameter[index] = old_value
    return (loss_plus - loss_minus) / (2 * epsilon)


rng = np.random.default_rng(7)
W1_test = rng.normal(0, 0.4, (2, 2)); b1_test = rng.normal(0, 0.1, 2)
W2_test = rng.normal(0, 0.4, (2, 2)); b2_test = rng.normal(0, 0.1, 2)
W3_test = rng.normal(0, 0.4, (2, 2)); b3_test = rng.normal(0, 0.1, 2)
x_test, y_test = X_data[10], y_data[10]


def current_loss():
    cache = forward_pass(x_test, W1_test, b1_test, W2_test, b2_test, W3_test, b3_test)
    return cross_entropy(cache['probs'], y_test)

cache_test = forward_pass(x_test, W1_test, b1_test, W2_test, b2_test, W3_test, b3_test)
grads_test = backward_pass(cache_test, y_test, W1_test, W2_test, W3_test)

analytical = grads_test['dL_dW1'][0, 0]
numerical = numerical_gradient(W1_test, (0, 0), current_loss)
print('Analytical gradient:', analytical)
print('Numerical gradient: ', numerical)
print('Absolute difference:', abs(analytical - numerical))
assert np.isclose(analytical, numerical, atol=1e-5)

Analytical gradient: -0.0718814937266061
Numerical gradient:  -0.07188149371839536
Absolute difference: 8.210737645342192e-12


## 7. Plot the transformations

In [8]:
def create_transformation_figure(X, y, params, activation_name, sample_idx):
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    W3, b3 = params['W3'], params['b3']
    activation, _ = get_activation_fn(activation_name)

    z1 = X @ W1 + b1
    a1 = activation(z1)
    z2 = a1 @ W2 + b2
    a2 = activation(z2)

    spaces = [
        (X, '1. Input space'),
        (z1, '2. Hidden 1: pre-activation z¹'),
        (a1, '3. Hidden 1: activation a¹'),
        (z2, '4. Hidden 2: pre-activation z²'),
        (a2, '5. Hidden 2: activation a²'),
    ]

    fig = make_subplots(rows=2, cols=3, subplot_titles=[title for _, title in spaces] + ['6. Decision boundary'])
    positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]

    for (space, _), (row, col) in zip(spaces, positions):
        for label in (0, 1):
            mask = y == label
            fig.add_trace(go.Scatter(
                x=space[mask, 0], y=space[mask, 1], mode='markers',
                marker={'size': 5, 'opacity': 0.6}, name=f'Class {label}',
                legendgroup=f'class-{label}', showlegend=(row == 1 and col == 1),
            ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=[space[sample_idx, 0]], y=[space[sample_idx, 1]], mode='markers',
            marker={'size': 15, 'symbol': 'star', 'line': {'width': 2}},
            name='Selected sample', legendgroup='selected',
            showlegend=(row == 1 and col == 1),
        ), row=row, col=col)

    x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
    y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    grid_probs = forward_pass(grid, W1, b1, W2, b2, W3, b3, activation_name)['probs']
    class_one_probability = grid_probs[:, 1].reshape(xx.shape)

    fig.add_trace(go.Contour(
        x=xx[0], y=yy[:, 0], z=class_one_probability,
        contours={'start': 0, 'end': 1, 'size': 0.1},
        opacity=0.55, showscale=True, colorbar={'title': 'P(class 1)'},
    ), row=2, col=3)

    for label in (0, 1):
        mask = y == label
        fig.add_trace(go.Scatter(
            x=X[mask, 0], y=X[mask, 1], mode='markers',
            marker={'size': 5, 'opacity': 0.7},
            legendgroup=f'class-{label}', showlegend=False,
        ), row=2, col=3)

    fig.add_trace(go.Scatter(
        x=[X[sample_idx, 0]], y=[X[sample_idx, 1]], mode='markers',
        marker={'size': 15, 'symbol': 'star', 'line': {'width': 2}},
        showlegend=False,
    ), row=2, col=3)

    fig.update_layout(height=780, title='How the network transforms the XOR problem', margin={'t': 90}, template='plotly_white')
    return fig

### Initial transformation preview

This figure is intentionally displayed **outside** the widget dashboard. Therefore, the input data and all intermediate spaces should appear immediately after running the cell.

In [9]:
preview_rng = np.random.default_rng(42)
preview_params = {
    'W1': preview_rng.normal(0, 0.8, (2, 2)),
    'b1': np.zeros(2),
    'W2': preview_rng.normal(0, 0.8, (2, 2)),
    'b2': np.zeros(2),
    'W3': preview_rng.normal(0, 0.8, (2, 2)),
    'b3': np.zeros(2),
}

initial_transformation_fig = create_transformation_figure(
    X_data,
    y_data,
    preview_params,
    activation_name='Tanh',
    sample_idx=100,
)

# Explicit Colab-compatible rendering.
initial_transformation_fig.show(renderer='colab' if IN_COLAB else None)


## 8. Formatting helpers

In [10]:
def array_text(value):
    return np.array2string(np.asarray(value), precision=4, suppress_small=True)


def matrix_html(value):
    arr = np.asarray(value)
    return f"<pre style='margin:4px 0 10px 0; white-space:pre-wrap'>{array_text(arr)}</pre>"


def forward_html(sample_idx, y_true, cache, loss):
    prediction = int(np.argmax(cache['probs']))
    return f"""
    <div style="border:1px solid #d1d5db;border-radius:10px;padding:16px;background:#ffffff">
      <h3 style="margin-top:0">Forward pass — sample {sample_idx}</h3>
      <b>Input x</b>{matrix_html(cache['x'])}
      <b>True class:</b> {y_true}<br><br>
      <b>Layer 1 pre-activation z¹</b>{matrix_html(cache['z1'])}
      <b>Layer 1 activation a¹</b>{matrix_html(cache['a1'])}
      <b>Layer 2 pre-activation z²</b>{matrix_html(cache['z2'])}
      <b>Layer 2 activation a²</b>{matrix_html(cache['a2'])}
      <b>Output logits z³</b>{matrix_html(cache['z3'])}
      <b>Softmax probabilities</b>{matrix_html(cache['probs'])}
      <b>Prediction:</b> {prediction}<br>
      <b>Cross-entropy loss:</b> {loss:.6f}
    </div>
    """


def backward_html(grads, learning_rate, params):
    gradient = grads['dL_dW1'][0, 0]
    next_w = params['W1'][0, 0] - learning_rate * gradient
    direction = 'increase' if gradient < 0 else 'decrease'
    return f"""
    <div style="border:1px solid #d1d5db;border-radius:10px;padding:16px;background:#ffffff">
      <h3 style="margin-top:0">Backward pass</h3>
      <h4>Output layer</h4>
      <b>∂L/∂z³</b>{matrix_html(grads['dL_dz3'])}
      <b>∂L/∂W³</b>{matrix_html(grads['dL_dW3'])}
      <b>∂L/∂b³</b>{matrix_html(grads['dL_db3'])}
      <h4>Hidden layer 2</h4>
      <b>∂L/∂a²</b>{matrix_html(grads['dL_da2'])}
      <b>∂L/∂z²</b>{matrix_html(grads['dL_dz2'])}
      <b>∂L/∂W²</b>{matrix_html(grads['dL_dW2'])}
      <b>∂L/∂b²</b>{matrix_html(grads['dL_db2'])}
      <h4>Hidden layer 1</h4>
      <b>∂L/∂a¹</b>{matrix_html(grads['dL_da1'])}
      <b>∂L/∂z¹</b>{matrix_html(grads['dL_dz1'])}
      <b>∂L/∂W¹</b>{matrix_html(grads['dL_dW1'])}
      <b>∂L/∂b¹</b>{matrix_html(grads['dL_db1'])}
      <h4>Example gradient-descent update</h4>
      <code>W1[0,0]new = W1[0,0] - η · ∂L/∂W1[0,0]</code><br><br>
      Current value: <b>{params['W1'][0,0]:.4f}</b><br>
      Gradient: <b>{gradient:.4f}</b><br>
      Direction: <b>{direction}</b><br>
      Updated value with η={learning_rate:.4f}: <b>{next_w:.4f}</b>
    </div>
    """


## 9. Interactive dashboard

Move any parameter and the six Plotly views will update. Below the plots, the complete numerical **forward pass** and **backward pass** remain visible at all times.

The **One gradient step** button applies

\[
\theta \leftarrow \theta - \eta\nabla_{\theta}L
\]

to the currently selected sample. **Train 20 steps** performs stochastic gradient descent using random samples.

In [11]:
rng = np.random.default_rng(42)
initial_params = {
    'W1': rng.normal(0, 0.5, (2, 2)), 'b1': np.zeros(2),
    'W2': rng.normal(0, 0.5, (2, 2)), 'b2': np.zeros(2),
    'W3': rng.normal(0, 0.5, (2, 2)), 'b3': np.zeros(2),
}

sample_widget = widgets.IntSlider(value=100, min=0, max=len(X_data)-1, step=1, description='Sample')
activation_widget = widgets.Dropdown(options=['Tanh', 'ReLU', 'Sigmoid', 'Leaky ReLU'], value='Tanh', description='Activation')
learning_rate_widget = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.1, description='η')

parameter_widgets = {}
for parameter_name, values in initial_params.items():
    for index in np.ndindex(values.shape):
        key = (parameter_name, index)
        index_label = ','.join(map(str, index))
        parameter_widgets[key] = widgets.FloatSlider(
            value=float(values[index]), min=-4, max=4, step=0.05,
            description=f'{parameter_name}[{index_label}]',
            continuous_update=False,
            layout=widgets.Layout(width='320px'),
            style={'description_width': '90px'},
        )


def params_from_widgets():
    params = {}
    for name, initial_value in initial_params.items():
        params[name] = np.empty_like(initial_value, dtype=float)
        for index in np.ndindex(initial_value.shape):
            params[name][index] = parameter_widgets[(name, index)].value
    return params


def write_params_to_widgets(params):
    for name, values in params.items():
        for index in np.ndindex(values.shape):
            parameter_widgets[(name, index)].value = float(np.clip(values[index], -4, 4))


plot_output = widgets.Output()
forward_output = widgets.Output()
backward_output = widgets.Output()
status_output = widgets.Output()


def evaluate_selected_sample():
    params = params_from_widgets()
    idx = sample_widget.value
    x, y_true = X_data[idx], y_data[idx]
    cache = forward_pass(x, **params, activation_name=activation_widget.value)
    loss = cross_entropy(cache['probs'], y_true)
    grads = backward_pass(cache, y_true, params['W1'], params['W2'], params['W3'], activation_widget.value)
    return params, idx, y_true, cache, loss, grads


def refresh(_=None):
    params, idx, y_true, cache, loss, grads = evaluate_selected_sample()

    with plot_output:
        clear_output(wait=True)
        fig = create_transformation_figure(X_data, y_data, params, activation_widget.value, idx)
        if IN_COLAB:
            # Embedding the Plotly HTML is more reliable than displaying its MIME bundle
            # from inside an ipywidgets.Output in Colab.
            display(HTML(pio.to_html(fig, include_plotlyjs='cdn', full_html=False)))
        else:
            display(fig)

    with forward_output:
        clear_output(wait=True)
        display(HTML(forward_html(idx, y_true, cache, loss)))

    with backward_output:
        clear_output(wait=True)
        display(HTML(backward_html(grads, learning_rate_widget.value, params)))


def apply_gradient_step(_):
    params, _, _, _, old_loss, grads = evaluate_selected_sample()
    eta = learning_rate_widget.value
    for name in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
        params[name] = params[name] - eta * grads[f'dL_d{name}']
    write_params_to_widgets(params)
    _, _, _, _, new_loss, _ = evaluate_selected_sample()
    with status_output:
        clear_output(wait=True)
        print(f'One gradient step: loss {old_loss:.6f} → {new_loss:.6f}')
    refresh()


def train_steps(_):
    params = params_from_widgets()
    eta = learning_rate_widget.value
    losses = []
    for _ in range(20):
        idx = int(rng.integers(0, len(X_data)))
        x, y_true = X_data[idx], y_data[idx]
        cache = forward_pass(x, **params, activation_name=activation_widget.value)
        losses.append(cross_entropy(cache['probs'], y_true))
        grads = backward_pass(cache, y_true, params['W1'], params['W2'], params['W3'], activation_widget.value)
        for name in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
            params[name] -= eta * grads[f'dL_d{name}']
    write_params_to_widgets(params)
    with status_output:
        clear_output(wait=True)
        print(f'20 SGD steps completed. Mean sampled loss: {np.mean(losses):.6f}')
    refresh()


def reset_parameters(_):
    write_params_to_widgets(initial_params)
    with status_output:
        clear_output(wait=True)
        print('Parameters reset.')
    refresh()


step_button = widgets.Button(description='One gradient step', button_style='success', icon='step-forward')
train_button = widgets.Button(description='Train 20 steps', button_style='info', icon='play')
reset_button = widgets.Button(description='Reset', button_style='warning', icon='refresh')
refresh_button = widgets.Button(description='Refresh', icon='sync')

step_button.on_click(apply_gradient_step)
train_button.on_click(train_steps)
reset_button.on_click(reset_parameters)
refresh_button.on_click(refresh)

for widget in [sample_widget, activation_widget, learning_rate_widget, *parameter_widgets.values()]:
    widget.observe(refresh, names='value')

layer1_controls = widgets.VBox([parameter_widgets[k] for k in parameter_widgets if k[0] in ('W1', 'b1')])
layer2_controls = widgets.VBox([parameter_widgets[k] for k in parameter_widgets if k[0] in ('W2', 'b2')])
output_controls = widgets.VBox([parameter_widgets[k] for k in parameter_widgets if k[0] in ('W3', 'b3')])

accordion = widgets.Accordion(children=[layer1_controls, layer2_controls, output_controls])
accordion.set_title(0, 'Layer 1 parameters')
accordion.set_title(1, 'Layer 2 parameters')
accordion.set_title(2, 'Output parameters')

control_panel = widgets.VBox([
    widgets.HTML('<h3>Controls</h3>'),
    sample_widget, activation_widget, learning_rate_widget,
    widgets.HBox([step_button, train_button]),
    widgets.HBox([reset_button, refresh_button]),
    status_output,
    accordion,
], layout=widgets.Layout(width='350px'))

pass_panels = widgets.VBox([
    widgets.HTML('<h2 style="margin:16px 0 8px">Numerical forward and backward passes</h2>'),
    forward_output,
    backward_output,
], layout=widgets.Layout(width='100%'))

right_panel = widgets.VBox([plot_output, pass_panels], layout=widgets.Layout(width='950px'))
dashboard = widgets.HBox([control_panel, right_panel], layout=widgets.Layout(align_items='flex-start', width='100%'))

display(dashboard)
refresh()

## 10. Suggested classroom explorations

1. Set a first-layer weight to zero. Which input direction disappears from the first hidden representation?
2. Compare **Tanh** and **ReLU** using the same parameters. Which gradients become zero?
3. Increase the magnitude of all weights. What happens to sigmoid gradients?
4. Select one sample, record its loss, and apply one gradient step. Does the selected-sample loss always decrease for every learning rate?
5. Train for several groups of 20 steps. Watch how the XOR pattern becomes easier to separate.

## Google Colab notes

1. Run the notebook with **Runtime → Run all**.
2. The setup cell enables Colab's custom widget manager automatically.
3. The static plots use `fig.show(renderer="colab")`, so they should appear without touching the controls.
4. The dashboard embeds Plotly as HTML inside the widget output because Colab can fail to render Plotly MIME output inside `ipywidgets.Output`.
5. If the dashboard controls do not appear, run the setup cell again and then rerun the dashboard cell.

6. In this version, the forward and backward passes are stacked below the plots instead of hidden inside tabs.